In [ ]:
%pip install -U mlflow langchain langchain-community langchain-text-splitters pypdf faiss-cpu sentence-transformers gradio openai databricks-langchain
dbutils.library.restartPython()

In [0]:
import os
import json
import glob
import mlflow
from datetime import datetime

PROJECT_NAME = "DinoRAG"
DOMAIN_NAME = "Dinosaurs"

CATALOG = "main"
SCHEMA = "default"
VOLUME = "dino_pdfs"

VOLUME_URI = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
PROJECT_URI = f"{VOLUME_URI}/project"
ARTIFACTS_URI = f"{PROJECT_URI}/artifacts"
INDEX_URI = f"{PROJECT_URI}/faiss_index"
CONFIG_URI = f"{PROJECT_URI}/config"

for uri in [PROJECT_URI, ARTIFACTS_URI, INDEX_URI, CONFIG_URI]:
    dbutils.fs.mkdirs(uri)

MLFLOW_EXPERIMENT = "/Shared/Dinosaur_RAG_Experiments"

project_config = {
    "project_name": PROJECT_NAME,
    "domain_name": DOMAIN_NAME,
    "catalog": CATALOG,
    "schema": SCHEMA,
    "volume": VOLUME,
    "volume_uri": VOLUME_URI,
    "project_uri": PROJECT_URI,
    "artifacts_uri": ARTIFACTS_URI,
    "index_uri": INDEX_URI,
    "config_uri": CONFIG_URI,
    "mlflow_experiment": MLFLOW_EXPERIMENT,
    "chunk_size": 800,
    "chunk_overlap": 150,
    "top_k": 4,
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "chat_endpoint": "databricks-meta-llama-3-1-8b-instruct",
    "delta_table": f"{CATALOG}.{SCHEMA}.dinosaur_chunks"
}

config_path = os.path.join(CONFIG_URI, "project_config.json")
with open(config_path, "w") as f:
    json.dump(project_config, f, indent=2)

print("Saved config to:", config_path)
print(json.dumps(project_config, indent=2))

Saved config to: /Volumes/main/default/dino_pdfs/project/config/project_config.json
{
  "project_name": "DinoRAG",
  "domain_name": "Dinosaurs",
  "catalog": "main",
  "schema": "default",
  "volume": "dino_pdfs",
  "volume_uri": "/Volumes/main/default/dino_pdfs",
  "project_uri": "/Volumes/main/default/dino_pdfs/project",
  "artifacts_uri": "/Volumes/main/default/dino_pdfs/project/artifacts",
  "index_uri": "/Volumes/main/default/dino_pdfs/project/faiss_index",
  "config_uri": "/Volumes/main/default/dino_pdfs/project/config",
  "mlflow_experiment": "/Shared/Dinosaur_RAG_Experiments",
  "chunk_size": 800,
  "chunk_overlap": 150,
  "top_k": 4,
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "chat_endpoint": "databricks-meta-llama-3-1-8b-instruct",
  "delta_table": "main.default.dinosaur_chunks"
}


In [0]:
VOLUME_URI = "/Volumes/main/default/dino_pdfs"

all_files = dbutils.fs.ls(VOLUME_URI)
pdf_files = [f.path for f in all_files if (not f.isDir()) and f.name.lower().endswith(".pdf")]

print(f"PDF folder: {VOLUME_URI}")
print(f"Found {len(pdf_files)} PDF file(s).")

for fp in pdf_files[:20]:
    print("-", fp)

PDF folder: /Volumes/main/default/dino_pdfs
Found 8 PDF file(s).
- dbfs:/Volumes/main/default/dino_pdfs/Document1_Dinosaur_Classification_and_General_Characteristics.pdf
- dbfs:/Volumes/main/default/dino_pdfs/Document2_History_of_Dinosaur_Discovery.pdf
- dbfs:/Volumes/main/default/dino_pdfs/Document3_Dinosaur_Classification_Size_and_Behavior.pdf
- dbfs:/Volumes/main/default/dino_pdfs/Document4_Dinosaur_Communication_Reproduction_and_Physiology.pdf
- dbfs:/Volumes/main/default/dino_pdfs/Document5_Dinosaur_Physiology_Behavior_and_Extinction_Events.pdf
- dbfs:/Volumes/main/default/dino_pdfs/Document6_Dinosaur_Reproduction_Parental_Care_and_Migration Behavior.pdf
- dbfs:/Volumes/main/default/dino_pdfs/Document7_Dinosaur_Sounds_Communication_and_Social_Interaction.pdf
- dbfs:/Volumes/main/default/dino_pdfs/Document8_Common_Misconceptions_About_Dinosaurs.pdf


In [0]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name="00_setup_validation"):
    mlflow.log_params({
        "project_name": PROJECT_NAME,
        "domain_name": DOMAIN_NAME,
        "catalog": CATALOG,
        "schema": SCHEMA,
        "volume": VOLUME,
        "embedding_model_name": project_config["embedding_model_name"],
        "chat_endpoint": project_config["chat_endpoint"]
    })
    mlflow.log_metric("num_pdf_files_found", len(pdf_files))
    mlflow.log_text(json.dumps(project_config, indent=2), "project_config.json")

print("MLflow experiment initialized:", MLFLOW_EXPERIMENT)

MLflow experiment initialized: /Shared/Dinosaur_RAG_Experiments


In [0]:
import pandas as pd

preview_rows = []
for fp in pdf_files:
    # Convert dbfs:/ URI to local path for os operations
    local_path = fp.replace('dbfs:', '')
    preview_rows.append({
        "file_name": os.path.basename(local_path),
        "size_kb": round(os.path.getsize(local_path) / 1024, 2)
    })

display(pd.DataFrame(preview_rows))

file_name,size_kb
Document1_Dinosaur_Classification_and_General_Characteristics.pdf,275.33
Document2_History_of_Dinosaur_Discovery.pdf,200.13
Document3_Dinosaur_Classification_Size_and_Behavior.pdf,199.27
Document4_Dinosaur_Communication_Reproduction_and_Physiology.pdf,172.82
Document5_Dinosaur_Physiology_Behavior_and_Extinction_Events.pdf,180.25
Document6_Dinosaur_Reproduction_Parental_Care_and_Migration Behavior.pdf,151.63
Document7_Dinosaur_Sounds_Communication_and_Social_Interaction.pdf,144.56
Document8_Common_Misconceptions_About_Dinosaurs.pdf,123.65
